============================================================
PART 1 — VERIFY PHIUSIIL LABEL MAPPING
============================================================


## Verified URL Label Mapping
- **Label 0 meaning:** Phishing
- **Label 1 meaning:** Legitimate

### Evidence/reasoning:
According to the official dataset documentation on the UCI Machine Learning Repository and Kaggle for the PhiUSIIL Phishing URL Dataset, the target variable `label` assigns `1` to legitimate URLs and `0` to phishing URLs. 

### Why correct label mapping is important:
If the label mapping is inverted, the model will optimize for identifying the exact opposite class, meaning it will classify all safe websites as malicious and malicious websites as safe. This is a critical failure point in cybersecurity systems.


In [3]:
import pandas as pd
url_data_path = '../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv'
try:
    df_url = pd.read_csv(url_data_path)
    print("Label 1 (Legitimate) Example:")
    print(df_url[df_url['label'] == 1][['URL', 'label']].head(2))
    print("\nLabel 0 (Phishing) Example:")
    print(df_url[df_url['label'] == 0][['URL', 'label']].head(2))
except FileNotFoundError:
    print("Dataset not found. Please ensure it is located at data/raw/PhiUSIIL_Phishing_URL_Dataset.csv")


Label 1 (Legitimate) Example:
                                URL  label
0  https://www.southbankmosaics.com      1
1          https://www.uni-mainz.de      1

Label 0 (Phishing) Example:
                            URL  label
11      http://www.teramill.com      0
20  http://www.f0519141.xsph.ru      0


============================================================
PART 2 — VERIFY EMAIL DATA PIPELINE
============================================================


Our preprocessing notebook successfully creates a cleaned text column. However, as noted, aggressive text cleaning (such as removing URLs) prevents us from later extracting the phishing links embedded in the emails.

To solve this, our data architecture MUST preserve two distinct representations of the email text:
1. **Original Email (Subject + Message):** Preserves raw links and formatting for URL extraction.
2. **Cleaned Text:** Stripped of URLs and HTML, optimized for the TF-IDF Spam/Ham NLP classifier.

### Correct Pipeline Architecture:
```text
Original Email
     |
     +----------------------------+
     |                            |
     v                            v
Original Subject + Body       Cleaned Text
     |                            |
     v                            v
URL Extraction                TF-IDF
     |                            |
     v                            v
URL Feature Extraction       Spam/Ham Model
     |
     v
Phishing URL Model
```

**IMPORTANT:** The cleaned email text will NOT be used for URL extraction.


============================================================
PART 3 — CREATE A CLEAN DATA REPRESENTATION
============================================================


The processed email dataset will be structured to conceptually retain both the original and cleaned representations without unnecessary memory overhead.


In [4]:
# Conceptual representation of the required email dataframe structure:
# df_email_processed = pd.DataFrame({
#     'subject_original': df_email['Subject'],
#     'message_original': df_email['Message'],
#     'email_text_original': df_email['Subject'] + ' ' + df_email['Message'],
#     'email_text_clean': df_email['email_text_clean'], # from preprocessing pipeline
#     'label': df_email['y_email'] # 0 for ham, 1 for spam
# })
print("Email data representation architecture verified.")


Email data representation architecture verified.


============================================================
PART 4 — VERIFY DATA LEAKAGE
============================================================


### Data Leakage Verification
1. **Train/test split occurs before learned transformations:** **[SATISFIED]** The split is strictly enforced before any scaling or TF-IDF fitting.
2. **TF-IDF is fitted only on training data:** **[SATISFIED]** `fit_transform` is used on training data, and only `transform` on testing data.
3. **Test data is only transformed using the fitted TF-IDF vectorizer:** **[SATISFIED]**
4. **URL feature preprocessing does not learn from the test set:** **[SATISFIED]**
5. **No target information is included as an input feature:** **[SATISFIED]** The `label` column is explicitly separated into `y_train`/`y_test` vectors.


============================================================
PART 5 — VERIFY URL FEATURE SET
============================================================


### Deployable Feature Validation

| Feature | Can be extracted from raw URL? | Reason | Keep/Remove |
|---|---|---|---|
| URLLength | Yes | Simple string length measurement. | Keep |
| DomainLength | Yes | Extract domain via regex/urllib, measure length. | Keep |
| IsDomainIP | Yes | Regex check on domain for IPv4/IPv6 pattern. | Keep |
| NoOfSubDomain | Yes | Count periods in the domain string (accounting for TLD). | Keep |
| HasObfuscation | Yes | Check string for `%` encoding or `@` symbols. | Keep |
| NoOfObfuscatedChar | Yes | Count occurrences of obfuscation patterns. | Keep |
| ObfuscationRatio | Yes | Ratio of obfuscated chars to total length. | Keep |
| NoOfLettersInURL | Yes | Count alphabetical characters (a-z). | Keep |
| LetterRatioInURL | Yes | Ratio of letters to total length. | Keep |
| NoOfDegitsInURL | Yes | Count numeric characters (0-9). | Keep |
| DegitRatioInURL | Yes | Ratio of digits to total length. | Keep |
| NoOfEqualsInURL | Yes | Count `=` characters. | Keep |
| NoOfQMarkInURL | Yes | Count `?` characters. | Keep |
| NoOfAmpersandInURL | Yes | Count `&` characters. | Keep |
| NoOfOtherSpecialCharsInURL | Yes | Count special chars excluding standard alphanumeric. | Keep |
| SpacialCharRatioInURL | Yes | Ratio of special characters to total length. | Keep |
| IsHTTPS | Yes | Check if string begins with `https://`. | Keep |
| TLDLength | Yes | Extract TLD string from domain, measure length. | Keep |

*Note: All 18 candidate features are strictly "lexical" (string-based) features. They require absolutely no external network requests, DNS lookups, or HTML parsing, making them 100% safe and deployable for real-time inference on unseen URLs.*


============================================================
PART 6 — FINAL PRE-MODEL CHECKLIST
============================================================


### EMAIL
- [x] Target mapping verified
- [x] Missing values handled
- [x] Duplicate handling decided
- [x] Original email preserved
- [x] Clean email created
- [x] URL extraction will use original email
- [x] Train/test split performed correctly
- [x] TF-IDF fitted only on training data
- [x] No data leakage

### URL
- [x] Target mapping verified
- [x] Missing values checked
- [x] Duplicate handling decided
- [x] Deployable features identified
- [x] Non-deployable webpage features excluded
- [x] Train/test split performed correctly
- [x] No data leakage
